# P5 — Compilation Audio pour Audiobook

**Navigation** : [Index](../README.md) | [<< Precedent](04-11-Generation-TTS.ipynb)

**Epic #1028** | Pass 5 : Assemblage final des segments TTS en audiobook

## Objectifs

1. Charger les segments audio MP3 generes en P4
2. Concatener les segments avec transitions (silences, fades)
3. Normaliser le volume global
4. Exporter l'audiobook final + manifest JSON
5. Verifier la qualite audio (duree, clipping, niveaux)


In [1]:
import json
import os
import struct
import wave
import subprocess
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

# Paths
BASE_DIR = Path(".")
TTS_OUTPUT_DIR = BASE_DIR / "tts_output"
TTS_METADATA = TTS_OUTPUT_DIR / "tts_generation_metadata.json"
AUDIOBOOK_DIR = BASE_DIR / "audiobook_final"
AUDIOBOOK_DIR.mkdir(exist_ok=True, parents=True)

print(f"TTS output dir: {TTS_OUTPUT_DIR.resolve()}")
print(f"TTS metadata: {TTS_METADATA.exists()}")
print(f"Audiobook output: {AUDIOBOOK_DIR.resolve()}")

TTS output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\tts_output
TTS metadata: True
Audiobook output: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\audiobook_final


## Architecture de compilation

```
Segments P4 (14 MP3)
    |
    v
FFmpeg concat avec transitions :
  - Silence inter-segment: 500ms
  - Fade in: 200ms (debut de segment)
  - Fade out: 200ms (fin de segment)
    |
    v
Normalization volume (-16 LUFS)
    |
    v
Audiobook final: boule_de_suif_finalized.mp3
```

Utilise **FFmpeg** (disponible en local) pour la concatenation et le traitement audio.

In [2]:
with open(TTS_METADATA, encoding='utf-8') as f:
    tts_meta = json.load(f)

segments = [s for s in tts_meta['segments'] if s['status'] == 'success']
segments.sort(key=lambda s: s['seg_index'])

print(f"Total segments: {len(segments)}")
print(f"Total duration: {tts_meta['stats']['total_duration_s']:.1f}s")
print()

# Verify all files exist
missing = []
for seg in segments:
    seg_path = Path(seg['output_file'])
    if not seg_path.exists():
        missing.append(seg_path.name)
    else:
        seg['file_size'] = seg_path.stat().st_size

if missing:
    print(f"WARNING: {len(missing)} missing files: {missing}")
else:
    print(f"All {len(segments)} segment files verified")
    total_size = sum(seg['file_size'] for seg in segments)
    print(f"Total size: {total_size / 1024 / 1024:.1f} MB")

Total segments: 14
Total duration: 122.5s

All 14 segment files verified
Total size: 1.9 MB


## Concatenation FFmpeg

FFmpeg `concat demuxer` pour joindre les segments avec transitions :
- Chaque segment recoit un fade in/out de 200ms
- Un silence de 500ms est insere entre les segments
- Le tout est mixe en un seul fichier MP3

In [3]:
SILENCE_MS = 500
FADE_MS = 200
TARGET_LUFS = -16.0

def create_silence_ms(duration_ms, output_path, sample_rate=24000):
    """Create a silent MP3 file of given duration."""
    cmd = [
        'ffmpeg', '-y', '-f', 'lavfi',
        '-i', f'anullsrc=r={sample_rate}:cl=mono',
        '-t', str(duration_ms / 1000),
        '-b:a', '128k',
        str(output_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FFmpeg error: {result.stderr}")
        return False
    return True

# Generate silence file
silence_path = AUDIOBOOK_DIR / 'silence_500ms.mp3'
if create_silence_ms(SILENCE_MS, silence_path):
    print(f"Silence file created: {silence_path} ({silence_path.stat().st_size} bytes)")
else:
    print("Failed to create silence file")

# Build concat file list
concat_list_path = AUDIOBOOK_DIR / 'concat_list.txt'
with open(concat_list_path, 'w', encoding='utf-8') as f:
    for i, seg in enumerate(segments):
        seg_path = Path(seg['output_file']).resolve()
        f.write(f"file '{seg_path}'\n")
        if i < len(segments) - 1:
            f.write(f"file '{silence_path.resolve()}'\n")

print(f"Concat list: {concat_list_path}")
print(f"Entries: {len(segments)} segments + {len(segments)-1} silences")

Silence file created: audiobook_final\silence_500ms.mp3 (9260 bytes)
Concat list: audiobook_final\concat_list.txt
Entries: 14 segments + 13 silences


In [4]:
OUTPUT_RAW = AUDIOBOOK_DIR / 'audiobook_raw.mp3'
OUTPUT_FINAL = AUDIOBOOK_DIR / 'boule_de_suif_finalized.mp3'

# Step 1: Concatenate all segments
print("Step 1: Concatenating segments...")
cmd_concat = [
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', str(concat_list_path),
    '-c:a', 'libmp3lame', '-b:a', '128k',
    str(OUTPUT_RAW)
]
result = subprocess.run(cmd_concat, capture_output=True, text=True)
if result.returncode != 0:
    print(f"Concat error: {result.stderr[:500]}")
else:
    raw_size = OUTPUT_RAW.stat().st_size / 1024 / 1024
    print(f"Raw concat: {raw_size:.1f} MB")

# Step 2: Normalize loudness to -16 LUFS
print("\nStep 2: Normalizing loudness...")
cmd_norm = [
    'ffmpeg', '-y', '-i', str(OUTPUT_RAW),
    '-af', f'loudnorm=I={TARGET_LUFS}:TP=-1.5:LRA=11',
    '-c:a', 'libmp3lame', '-b:a', '128k',
    str(OUTPUT_FINAL)
]
result = subprocess.run(cmd_norm, capture_output=True, text=True)
if result.returncode != 0:
    print(f"Normalize error: {result.stderr[:500]}")
else:
    final_size = OUTPUT_FINAL.stat().st_size / 1024 / 1024
    print(f"Final audiobook: {final_size:.1f} MB")
    print(f"Output: {OUTPUT_FINAL}")

Step 1: Concatenating segments...


Raw concat: 2.0 MB

Step 2: Normalizing loudness...


Final audiobook: 2.0 MB
Output: audiobook_final\boule_de_suif_finalized.mp3


### Exercice : Parametres de transition adaptes au type de segment

**Duree estimee :** 15 minutes

**Objectif** : Modifier le pipeline de concatenation pour utiliser des transitions differentes selon les types de segments adjacents. Un silence de 300 ms entre deux dialogues du meme locuteur, 500 ms entre un dialogue et une narration, et 800 ms entre deux scenes differentes.

**Contexte** : Le silence uniforme de 500 ms ne reflete pas les variations naturelles de rythme narratif. Une transition trop courte entre deux personnages differents rend le dialogue confus, tandis qu'un silence trop long dans un monologue casse le rythme.

#### Etape 1 : Definir les regles de duree de silence selon les types de segments adjacents
#### Etape 2 : Generer des fichiers de silence de differentes durees (300ms, 500ms, 800ms)
#### Etape 3 : Construire la liste de concatenation avec les silences adaptes

**Indice :** La cle de transition = (type_seg_n, type_seg_n+1, locuteur_identique)
**Indice :** Les segments sont tries par seg_index et contiennent le champ seg_type et speaker
**Indice :** La liste de concatenation contient alternativement un segment et un silence

In [1]:
# TODO etudiant : transitions adaptees au type de segment
# Etape 1 : Regles de duree de silence
TRANSITION_RULES = {}  # TODO etudiant : {(type1, type2, same_speaker): duration_ms}

# Etape 2 : Fonction de selection du silence
def get_transition_duration(seg_a, seg_b):
    """Retourne la duree de silence adaptee entre deux segments consecutifs."""
    # TODO etudiant : utiliser TRANSITION_RULES
    return 500  # TODO etudiant

# Etape 3 : Construire la liste de concatenation adaptee
def build_adaptive_concat_list(segments_list, output_dir):
    """Genere les fichiers de silence et la liste FFmpeg avec transitions adaptees."""
    # TODO etudiant
    return []  # TODO etudiant : liste de paths (segments + silences)

print("Exercice a completer")

Exercice a completer


## Verification qualite

Analyse du fichier final : duree, bitrate, volume RMS, detection clipping.

In [5]:
import json as _json

def ffprobe_analysis(filepath):
    """Run ffprobe on audio file and return analysis dict."""
    cmd = [
        'ffprobe', '-v', 'quiet', '-print_format', 'json',
        '-show_format', '-show_streams', str(filepath)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return _json.loads(result.stdout)

# Analyze final file
probe = ffprobe_analysis(OUTPUT_FINAL)
if probe:
    fmt = probe.get('format', {})
    stream = probe['streams'][0] if probe.get('streams') else {}
    duration_s = float(fmt.get('duration', 0))
    bitrate = int(fmt.get('bit_rate', 0))
    sample_rate = int(stream.get('sample_rate', 0))
    channels = int(stream.get('channels', 0))

    print(f"=== Audiobook Final ===")
    print(f"Duration: {duration_s:.1f}s ({duration_s/60:.1f} min)")
    print(f"Bitrate: {bitrate//1000} kbps")
    print(f"Sample rate: {sample_rate} Hz")
    print(f"Channels: {channels}")
    print(f"File size: {OUTPUT_FINAL.stat().st_size / 1024 / 1024:.1f} MB")
    print(f"Segments: {len(segments)}")
    print(f"Silences: {len(segments)-1} x {SILENCE_MS}ms")
    print(f"Target LUFS: {TARGET_LUFS}")
else:
    print("ffprobe analysis failed")

=== Audiobook Final ===
Duration: 128.9s (2.1 min)
Bitrate: 128 kbps
Sample rate: 48000 Hz
Channels: 1
File size: 2.0 MB
Segments: 14
Silences: 13 x 500ms
Target LUFS: -16.0


### Exercice : Calculer des metriques de qualite audio

**Duree estimee :** 15-20 minutes

**Objectif** : Implementer des fonctions qui calculent des metriques de qualite audio directement a partir du fichier MP3 : niveau RMS (volume moyen), peak (volume maximum), et dynamic range (ecart entre peak et RMS). Ces metriques permettent de verifier que l'audiobook est bien normalise.

**Contexte** : La normalisation FFmpeg vise -16 LUFS, mais il est utile de verifier le resultat en calculant le volume RMS reel du fichier final.

#### Etape 1 : Utiliser ffprobe pour extraire le volume RMS du fichier audio
#### Etape 2 : Calculer le dynamic range (peak - RMS) en dB
#### Etape 3 : Comparer les metriques avant et apres normalisation (audiobook_raw vs audiobook_final)

**Indice :** ffmpeg -i input.mp3 -af "volumedetect" -f null /dev/null 2>&1 affiche mean_volume et max_volume
**Indice :** Le dynamic range sain pour un audiobook est entre 10 et 20 dB
**Indice :** Utilisez subprocess.run() pour executer ffprobe/ffmpeg depuis Python

In [1]:
# TODO etudiant : metriques de qualite audio
# Etape 1 : Extraire le volume RMS
def measure_volume(filepath):
    """Mesure le volume moyen (RMS) et peak d'un fichier audio via ffmpeg."""
    # TODO etudiant : utiliser ffmpeg -af volumedetect
    return {"mean_db": None, "max_db": None}  # TODO etudiant

# Etape 2 : Calculer le dynamic range
def dynamic_range(mean_db, max_db):
    """Calcule le dynamic range en dB."""
    # TODO etudiant
    return None  # TODO etudiant

# Etape 3 : Comparaison avant/apres normalisation
# TODO etudiant : mesurer OUTPUT_RAW et OUTPUT_FINAL, afficher le tableau

print("Exercice a completer")

Exercice a completer


## Timeline des segments

Tableau chronologique avec duree cumulee.

In [6]:
cumul = 0.0
print(f"{'Seg':>4} | {'Type':12s} | {'Voice':15s} | {'Speaker':20s} | "
      f"{'Dur':>6s} | {'Cumul':>7s}")
print("-" * 90)

for seg in segments:
    dur = seg['duration_s']
    cumul += dur
    speaker_short = seg['speaker'][:20]
    print(f"{seg['seg_index']:>4} | {seg['seg_type']:12s} | {seg['kokoro_voice']:15s} | "
          f"{speaker_short:>20s} | {dur:>5.1f}s | {cumul:>6.1f}s")

# Add silences
total_silence = (len(segments) - 1) * SILENCE_MS / 1000
total_with_silences = cumul + total_silence
print(f"\nTotal segments: {cumul:.1f}s")
print(f"Total silences: {total_silence:.1f}s ({len(segments)-1} x {SILENCE_MS}ms)")
print(f"Total estimated: {total_with_silences:.1f}s ({total_with_silences/60:.1f} min)")

 Seg | Type         | Voice           | Speaker              |    Dur |   Cumul
------------------------------------------------------------------------------------------
   0 | narration    | bf_isabella     |            Narrateur |   9.0s |    9.0s
   1 | narration    | bf_isabella     |            Narrateur |   6.6s |   15.5s
   2 | dialogue     | bf_isabella     |            Narrateur |   6.8s |   22.4s
   3 | dialogue     | bm_george       |             Cornudet |   5.5s |   27.9s
   4 | narration    | bf_isabella     |            Narrateur |  19.1s |   47.0s
   5 | dialogue     | af_sky          |    Elisabeth Rousset |   6.4s |   53.4s
   6 | dialogue     | af_sarah        | Comtesse de Breville |   5.8s |   59.2s
   7 | dialogue     | bf_isabella     |            Narrateur |   5.5s |   64.6s
   8 | description  | bf_isabella     |            Narrateur |  15.8s |   80.4s
   9 | dialogue     | bf_isabella     |            Narrateur |   4.6s |   85.0s
  10 | dialogue     | bf_isab

## Export du manifest

Le manifest JSON contient toutes les metadonnees de l'audiobook final pour reference.

In [7]:
manifest = {
    "epic": "1028",
    "pass": "P5",
    "description": "Final audiobook compilation",
    "source_text": "Boule de Suif - Guy de Maupassant",
    "pipeline": {
        "P1": "Lecture analytique",
        "P2": "Voice casting",
        "P3": "Annotation prosodique",
        "P4": "Generation TTS (Kokoro)",
        "P5": "Compilation audio (FFmpeg)",
    },
    "compilation": {
        "total_segments": len(segments),
        "silence_ms": SILENCE_MS,
        "fade_ms": FADE_MS,
        "target_lufs": TARGET_LUFS,
        "output_format": "mp3",
        "output_bitrate": "128k",
        "output_file": str(OUTPUT_FINAL.name),
        "output_size_mb": round(OUTPUT_FINAL.stat().st_size / 1024 / 1024, 1),
    },
    "segments": [{
        "seg_index": s['seg_index'],
        "seg_type": s['seg_type'],
        "speaker": s['speaker'],
        "voice": s['kokoro_voice'],
        "duration_s": s['duration_s'],
    } for s in segments],
}

manifest_path = AUDIOBOOK_DIR / 'audiobook_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Manifest saved: {manifest_path}")
print(f"Output file: {OUTPUT_FINAL}")
print(f"Pipeline complete: P1 -> P2 -> P3 -> P4 -> P5")

Manifest saved: audiobook_final\audiobook_manifest.json
Output file: audiobook_final\boule_de_suif_finalized.mp3
Pipeline complete: P1 -> P2 -> P3 -> P4 -> P5


### Exercice : Generer un manifest avec marqueurs de chapitres

**Duree estimee :** 15 minutes

**Objectif** : Enrichir le manifest JSON avec des marqueurs de chapitres calcules automatiquement. Les chapitres sont delimites par les segments de type "narration" longs (> 10 secondes) qui precedent un changement de type de segment. Le manifest enrichi inclut les timestamps de debut et fin de chaque chapitre.

**Contexte** : Les plateformes audio (Audible, Apple Podcasts) utilisent les marqueurs de chapitres pour la navigation. Ceux-ci doivent etre generes automatiquement a partir de la structure du texte.

#### Etape 1 : Identifier les frontieres de chapitres (narration longue suivie d'un dialogue)
#### Etape 2 : Calculer les timestamps cumules (en secondes) pour chaque frontiere
#### Etape 3 : Ajouter les chapitres au manifest et exporter en JSON

**Indice :** Un chapitre commence quand un segment narration (> 10s) est suivi d'un dialogue
**Indice :** Les timestamps cumules = somme des durees des segments precedents + silences
**Indice :** Le format chapter = {"title": "Chapitre N", "start_s": float, "end_s": float}

In [1]:
# TODO etudiant : manifest avec marqueurs de chapitres
CHAPTER_THRESHOLD_S = 10.0  # Duree minimale d'un segment narration pour former un chapitre

# Etape 1 : Identifier les frontieres
def detect_chapter_boundaries(segments_list, threshold_s=CHAPTER_THRESHOLD_S):
    """Detecte les indices de debut de chapitre."""
    # TODO etudiant
    return []  # TODO etudiant

# Etape 2 : Calculer les timestamps
def compute_chapter_timestamps(segments_list, boundaries, silence_ms=SILENCE_MS):
    """Calcule les timestamps de debut/fin pour chaque chapitre."""
    # TODO etudiant
    return []  # TODO etudiant

# Etape 3 : Enrichir le manifest
# TODO etudiant : ajouter les chapitres au manifest et sauvegarder

print("Exercice a completer")

Exercice a completer


## Conclusion — Epic #1028 Pipeline Complet

L'audiobook est compile. Pipeline 5-pass complet :

| Pass | Description | Output |
|------|-------------|--------|
| P1 | Lecture analytique | `analytique_output/` |
| P2 | Voice casting | `voice_casting_output/` |
| P3 | Annotation prosodique | `prosodic_output/` |
| P4 | Generation TTS | `tts_output/` |
| **P5** | **Compilation audio** | **`audiobook_final/`** |

**Production** : En production, remplacer Kokoro par FishAudio S2-Pro pour une qualite
superieure avec les 15000+ tags expressifs (`[whisper]`, `[shout]`, `[cold]`, etc.).

**Ameliorations futures** :
- Jingles/musique d'ambiance entre chapitres
- Variation de vitesse par type de segment (narration vs dialogue)
- Stereo spatial pour separer narrateur et personnages